# The same as the last 2 cells of BEST_MODEL.
Unfortunately dont have the best threshold anymore :\( pl retrain if you want

In [ ]:
import pandas as pd
import numpy as np

TEST_TSV = "../dontpatronizeme/semeval-2022/TEST/task4_test.tsv"

rows=[]
with open(TEST_TSV) as f:
    for line in f.readlines()[4:]:
        par_id=line.strip().split('\t')[0]
        art_id = line.strip().split('\t')[1]
        keyword=line.strip().split('\t')[2]
        country=line.strip().split('\t')[3]
        t=line.strip().split('\t')[4]#.lower()
        rows.append(
            {'par_id':par_id,
            'doc_id':art_id,
            'keyword':keyword,
            'country':country,
            'text':t, 
            }
        )
df_test = pd.DataFrame(rows, columns=['par_id', 'doc_id', 'keyword', 'country', 'text'])


In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments
)

MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def make_infer_ds(df_, max_len=256):
    ds = Dataset.from_pandas(df_[["text"]], preserve_index=False)

    def tok(b):
        return tokenizer(b["text"], truncation=True, max_length=max_len)

    return ds.map(tok, batched=True, remove_columns=["text"])


@torch.no_grad()
def predict_probs_pos(model, infer_ds, batch_size=16):
    model.eval()
    dl = DataLoader(infer_ds, batch_size=batch_size, shuffle=False, collate_fn=base_collator)
    probs = []

    for batch in dl:
        batch = {k: v.to(model.device) for k, v in batch.items()}
        logits = model(**batch).logits
        p = torch.softmax(logits, dim=-1)[:, 1]
        probs.append(p.detach().cpu())

    return torch.cat(probs).numpy()

In [ ]:
import torch
assert df_test["par_id"].is_unique, "Test TSV has duplicate par_id rows"
assert df_test["text"].notna().all(), "Some test texts are missing"

MAX_LEN=256
test_infer = make_infer_ds(df_test, MAX_LEN)

dev_probs_seeds = []
test_probs_seeds = []

device = "cuda" if torch.cuda.is_available() else "cpu"

for d in ["../checkpoints/task1_refit_seed42_w0.75_lr2e-05_ep4", "../checkpoints/task1_refit_seed43_w0.5_lr3e-05_ep4", "../checkpoints/task1_refit_seed44_w0.35_lr3e-05_ep4"]:
    m = AutoModelForSequenceClassification.from_pretrained(d).to(device)
    test_probs_seeds.append(predict_probs_pos(m, test_infer))

dev_probs_ens = np.mean(np.stack(dev_probs_seeds, axis=0), axis=0)
test_probs_ens = np.mean(np.stack(test_probs_seeds, axis=0), axis=0)
best_t = 0.1358598917722702
test_pred = (test_probs_ens >= best_t).astype(int)


with open("test.txt", "w", encoding="utf-8") as f:
    for p in test_pred.tolist():
        f.write(f"{int(p)}\n")

print("Wrote dev.txt and test.txt")